# Parte 1 – SQL (Databricks)
Este notebook contém as soluções dos exercícios:

- **1.1 Campeonato (times / jogos + classificação por pontos)**
- **1.2 Comissões (vendedores que atingem 1024 em até 3 comissões)**
- **1.3 Organização Empresarial (chefe indireto mais próximo com salário ≥ 2x)**

Padrão aplicado:
- `CREATE TABLE IF NOT EXISTS` (Delta)
- `MERGE` para carga idempotente
- consultas organizadas e fáceis de explicar


## 0) Parâmetros (Catalog / Schema)
Centraliza onde as tabelas serão criadas.


In [ ]:
# ===== PARAMETROS UC =====
CATALOG = "workspace"
SCHEMA = "cantustore"


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print("Schema ativo:", f"{CATALOG}.{SCHEMA}")


# 1.1 Campeonato
## 1) Criar tabelas (idempotente)


In [0]:
%sql
CREATE TABLE IF NOT EXISTS times (
  time_id   INT NOT NULL,
  time_nome STRING NOT NULL
)
USING DELTA;

CREATE TABLE IF NOT EXISTS jogos (
  jogo_id         INT NOT NULL,
  mandante_time   INT NOT NULL,
  visitante_time  INT NOT NULL,
  mandante_gols   INT NOT NULL,
  visitante_gols  INT NOT NULL
)
USING DELTA;


## 2) Carga idempotente via MERGE
Roda quantas vezes quiser sem duplicar.


In [0]:
%sql
MERGE INTO times AS tgt
USING (
  SELECT * FROM VALUES
    (10, 'Financeiro'),
    (20, 'Marketing'),
    (30, 'Logística'),
    (40, 'TI'),
    (50, 'Dados')
  AS src(time_id, time_nome)
) AS src
ON tgt.time_id = src.time_id
WHEN MATCHED THEN UPDATE SET
  tgt.time_nome = src.time_nome
WHEN NOT MATCHED THEN INSERT (time_id, time_nome)
VALUES (src.time_id, src.time_nome);

MERGE INTO jogos AS tgt
USING (
  SELECT * FROM VALUES
    (1, 30, 20, 1, 0),
    (2, 10, 20, 1, 2),
    (3, 20, 50, 2, 2),
    (4, 10, 30, 1, 0),
    (5, 30, 50, 0, 1)
  AS src(jogo_id, mandante_time, visitante_time, mandante_gols, visitante_gols)
) AS src
ON tgt.jogo_id = src.jogo_id
WHEN MATCHED THEN UPDATE SET
  tgt.mandante_time  = src.mandante_time,
  tgt.visitante_time = src.visitante_time,
  tgt.mandante_gols  = src.mandante_gols,
  tgt.visitante_gols = src.visitante_gols
WHEN NOT MATCHED THEN INSERT (jogo_id, mandante_time, visitante_time, mandante_gols, visitante_gols)
VALUES (src.jogo_id, src.mandante_time, src.visitante_time, src.mandante_gols, src.visitante_gols);


## 3) Classificação por pontos
Regras:
- vitória = 3
- empate = 1
- derrota = 0

Ordenação:
- `num_pontos` desc
- em empate, `time_id` asc


In [0]:
%sql
WITH resultados AS (
  -- pontos do mandante
  SELECT
    mandante_time AS time_id,
    CASE
      WHEN mandante_gols > visitante_gols THEN 3
      WHEN mandante_gols = visitante_gols THEN 1
      ELSE 0
    END AS pontos
  FROM jogos

  UNION ALL

  -- pontos do visitante
  SELECT
    visitante_time AS time_id,
    CASE
      WHEN visitante_gols > mandante_gols THEN 3
      WHEN visitante_gols = mandante_gols THEN 1
      ELSE 0
    END AS pontos
  FROM jogos
),
pontos_por_time AS (
  SELECT
    time_id,
    SUM(pontos) AS num_pontos
  FROM resultados
  GROUP BY time_id
)
SELECT
  t.time_id,
  t.time_nome,
  COALESCE(p.num_pontos, 0) AS num_pontos
FROM times t
LEFT JOIN pontos_por_time p
  ON t.time_id = p.time_id
ORDER BY num_pontos DESC, t.time_id ASC;


# 1.2 Comissões
## 1) Criar tabela + carga idempotente


In [0]:
%sql
CREATE TABLE IF NOT EXISTS comissoes (
  comprador STRING NOT NULL,
  vendedor  STRING NOT NULL,
  dataPgto  DATE   NOT NULL,
  valor     DOUBLE NOT NULL
)
USING DELTA;

MERGE INTO comissoes AS tgt
USING (
  SELECT * FROM VALUES
    ('Leonardo','Bruno'  , DATE '2000-01-01', 200.00),
    ('Leonardo','Matheus', DATE '2003-09-27',1024.00),
    ('Leonardo','Lucas'  , DATE '2006-06-26', 512.00),
    ('Marcos'  ,'Lucas'  , DATE '2020-12-17', 100.00),
    ('Marcos'  ,'Lucas'  , DATE '2002-03-22',  10.00),
    ('Cinthia' ,'Lucas'  , DATE '2021-03-20', 500.00),
    ('Mateus'  ,'Bruno'  , DATE '2007-06-02', 400.00),
    ('Mateus'  ,'Bruno'  , DATE '2006-06-26', 400.00),
    ('Mateus'  ,'Bruno'  , DATE '2015-06-26', 200.00)
  AS src(comprador, vendedor, dataPgto, valor)
) AS src
ON  tgt.comprador = src.comprador
AND tgt.vendedor  = src.vendedor
AND tgt.dataPgto   = src.dataPgto
AND tgt.valor      = src.valor
WHEN NOT MATCHED THEN
  INSERT (comprador, vendedor, dataPgto, valor)
  VALUES (src.comprador, src.vendedor, src.dataPgto, src.valor);


## 2) Query: vendedores que atingem **1024** com **até 3 transferências**
Ideia:
- Se existe algum conjunto de ≤ 3 comissões somando ≥ 1024,
- então a soma das **3 maiores comissões** desse vendedor também será ≥ 1024.
Logo basta:
- rankear por valor desc
- somar `top 3`
- filtrar `>= 1024`


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_top3_por_vendedor AS
WITH ranked AS (
  SELECT
    vendedor,
    valor,
    ROW_NUMBER() OVER (PARTITION BY vendedor ORDER BY valor DESC) AS rn
  FROM comissoes
)
SELECT
  vendedor,
  SUM(valor) AS soma_top3
FROM ranked
WHERE rn <= 3
GROUP BY vendedor;

SELECT vendedor
FROM vw_top3_por_vendedor
WHERE soma_top3 >= 1024
ORDER BY vendedor ASC;


# 1.3 Organização Empresarial
## 1) Criar tabela + carga idempotente


In [0]:
%sql
CREATE TABLE IF NOT EXISTS colaboradores (
  id       INT    NOT NULL,
  nome     STRING NOT NULL,
  salario  INT    NOT NULL,
  lider_id INT
)
USING DELTA;

MERGE INTO colaboradores AS tgt
USING (
  SELECT * FROM VALUES
    (40, 'Helen'   , 1500, 50),
    (50, 'Bruno'   , 3000, 10),
    (10, 'Leonardo', 4500, 20),
    (20, 'Marcos'  ,10000, CAST(NULL AS INT)),
    (70, 'Mateus'  , 1500, 10),
    (60, 'Cinthia' , 2000, 70),
    (30, 'Wilian'  , 1501, 50)
  AS src(id, nome, salario, lider_id)
) AS src
ON tgt.id = src.id
WHEN MATCHED THEN UPDATE SET
  tgt.nome     = src.nome,
  tgt.salario  = src.salario,
  tgt.lider_id = src.lider_id
WHEN NOT MATCHED THEN INSERT (id, nome, salario, lider_id)
VALUES (src.id, src.nome, src.salario, src.lider_id);


## 2) Query: chefe indireto mais próximo que ganha ≥ 2x
Objetivo:
Para cada funcionário, achar o primeiro chefe na cadeia (chefe direto, chefe do chefe, etc.)
que satisfaça:

`chefe.salario >= 2 * funcionario.salario`

Se nenhum satisfizer, retorna `NULL`.

Como resolvemos:
1. `CTE RECURSIVE` monta a cadeia funcionário → ancestrais com nível (distância).
2. Filtramos apenas chefes que ganham ≥ 2x.
3. Escolhemos o de menor nível (mais próximo).


In [0]:
%sql
WITH RECURSIVE hierarquia AS (
  -- Nível 1: chefe direto do funcionário
  SELECT
    c.id        AS id_funcionario,
    c.lider_id  AS id_chefe,
    1           AS nivel
  FROM colaboradores c
  WHERE c.lider_id IS NOT NULL

  UNION ALL

  -- Subindo na hierarquia: chefe do chefe, etc.
  SELECT
    h.id_funcionario,
    chefe.lider_id AS id_chefe,
    h.nivel + 1    AS nivel
  FROM hierarquia h
  JOIN colaboradores chefe
    ON h.id_chefe = chefe.id
  WHERE chefe.lider_id IS NOT NULL
),

chefes_aptos AS (
  -- Mantém apenas chefes que ganham pelo menos o dobro do funcionário
  SELECT
    h.id_funcionario,
    h.id_chefe,
    h.nivel
  FROM hierarquia h
  JOIN colaboradores func
    ON h.id_funcionario = func.id
  JOIN colaboradores chefe
    ON h.id_chefe = chefe.id
  WHERE chefe.salario >= 2 * func.salario
),

chefe_mais_proximo AS (
  -- Escolhe o chefe apto mais próximo (menor nível)
  SELECT
    id_funcionario,
    id_chefe,
    ROW_NUMBER() OVER (
      PARTITION BY id_funcionario
      ORDER BY nivel ASC
    ) AS ordem
  FROM chefes_aptos
)

-- Resultado final
SELECT
  f.id AS id_funcionario,
  c.id_chefe AS id_chefe
FROM colaboradores f
LEFT JOIN chefe_mais_proximo c
  ON f.id = c.id_funcionario
 AND c.ordem = 1
ORDER BY f.id;
